In [3]:
# ==============================================================================
# ESG_Scoring_hybrid_v7.ipynb — 5개 기업 순차 자동 채점 파이프라인
# 셀 1️⃣: 라이브러리 설치
# ==============================================================================
!pip install -q langchain-chroma langchain-google-genai chromadb kiwipiepy rank_bm25 google-genai openpyxl pandas numpy tqdm


In [4]:
# ------------------------------------------------------------------------------
# 셀 2️⃣: import + Gemini API 및 모델 설정
# ------------------------------------------------------------------------------
import json
import os
import re
import shutil
from getpass import getpass
from datetime import datetime

import numpy as np
import pandas as pd
from tqdm import tqdm
from kiwipiepy import Kiwi
import openpyxl

from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from google import genai
from google.genai import types

# doc_type -> 근거 인용에 쓸 한글 문서 라벨
DOC_TYPE_LABEL = {
    "business_report": "사업보고서",
    "governance_report": "지배구조보고서",
    "sustainability_report": "지속가능경영보고서",
}

# API 키 입력
GEMINI_API_KEY = os.environ.get("GOOGLE_API_KEY") or getpass("Gemini API Key: ")
os.environ["GOOGLE_API_KEY"] = GEMINI_API_KEY
client = genai.Client(api_key=GEMINI_API_KEY)

# 임베딩 & 채점 모델 정의
EMBEDDING_MODEL = "gemini-embedding-001"
JUDGE_MODEL = "gemini-3.1-flash-lite"  # 사용자 지정 (Gemini 3.1 Flash-Lite)

embeddings = GoogleGenerativeAIEmbeddings(model=EMBEDDING_MODEL)
kiwi = Kiwi()
RRF_K = 60


In [5]:
# ------------------------------------------------------------------------------
# 셀 3️⃣: 경로 설정 (esg_query_interface2 기준 통합)
# ------------------------------------------------------------------------------
REPORTS_ROOT = "/Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/05_채점 RAG/260902_RAG테스트/보고서 3종"

# [SK하이닉스 전용] build_rag_index_v6_production_SKHY.ipynb가 재인덱싱한
# SK하이닉스 전용 Chroma를 가리킴 (공용 인덱스와 분리돼 있음).
SKHY_CONTENT_ROOT = f"{REPORTS_ROOT}/SK하이닉스(000660)/content"
CHROMA_LOCAL = f"{SKHY_CONTENT_ROOT}/chroma_db_local"
EXTRACTION_DIAG_CSV = f"{SKHY_CONTENT_ROOT}/extraction_diagnostics.csv"

# [SK하이닉스 전용] 이 노트북은 SK하이닉스 하나만 채점 — 반도체(KSIC 261) 기업이라
# "반도체특화" 항목도 반드시 포함해서 채점해야 함.
COMPANY_CODE = "000660"
COMPANY_NAME = "SK하이닉스"
IS_C261 = True

RUBRIC_SRC = "/Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/05_채점 RAG/260902_RAG테스트/채점데이터셋/ESG_핵심지표_채점기준표_전체.xlsx"
MASTER_SHEET = "01_ESG통합진단마스터"
LOOKUP_SHEET = "02_점수산정참고표"
HEADER_ROW = 1
DATA_START_ROW = 2


In [6]:
# ------------------------------------------------------------------------------
# 셀 4️⃣: 검증된 FilteredHybridRetriever 클래스 및 쿼리 캐시 정의
# ------------------------------------------------------------------------------
KIWI_KEEP_TAGS = {"NNG", "NNP", "NNB", "VV", "VA", "SL", "SN", "SH"}

def kiwi_tokenize(text: str):
    return [t.form for t in kiwi.tokenize(str(text)) if t.tag in KIWI_KEEP_TAGS]

class FilteredHybridRetriever:
    """공동 인터페이스(interface2)에서 버그 수정 및 FIFO 캐시가 검증된 검색기"""
    def __init__(self, vectorstore, tokenize_func, k=30, weights=(0.5, 0.5), max_cache_entries=30):
        self.vectorstore = vectorstore
        self.tokenize_func = tokenize_func
        self.k = k
        self.weights = weights
        self._bm25_cache: dict[str, object] = {}
        self._max_cache_entries = max_cache_entries

    def _get_bm25_retriever(self, filter_dict):
        # json 직렬화를 통한 unhashable TypeError 방지 (버그픽스)
        sig = json.dumps(filter_dict, sort_keys=True, ensure_ascii=False) if filter_dict else "all"
        if sig in self._bm25_cache:
            return self._bm25_cache[sig]
        raw = self.vectorstore.get(where=filter_dict if filter_dict else None, include=["documents", "metadatas"])
        docs = [Document(page_content=d, metadata=m) for d, m in zip(raw["documents"], raw["metadatas"])]
        if not docs:
            self._set_cache(sig, None)
            return None
        retriever = BM25Retriever.from_documents(docs, preprocess_func=self.tokenize_func)
        retriever.k = self.k
        self._set_cache(sig, retriever)
        return retriever

    def _set_cache(self, sig, value):
        if len(self._bm25_cache) >= self._max_cache_entries:
            oldest_key = next(iter(self._bm25_cache))
            del self._bm25_cache[oldest_key]
        self._bm25_cache[sig] = value

    def invoke(self, query: str, filter: dict | None = None, top_n: int = 5, query_embedding: list[float] | None = None):
        if query_embedding is not None:
            dense_docs = self.vectorstore.similarity_search_by_vector(query_embedding, k=self.k, filter=filter)
        else:
            dense_docs = self.vectorstore.similarity_search(query, k=self.k, filter=filter)

        bm25_retriever = self._get_bm25_retriever(filter)
        bm25_docs = bm25_retriever.invoke(query) if bm25_retriever else []

        scores: dict[str, float] = {}
        lookup: dict[str, Document] = {}

        for rank, doc in enumerate(dense_docs):
            key = doc.metadata.get("chunk_id", doc.page_content)
            scores[key] = scores.get(key, 0.0) + self.weights[0] * (1.0 / (RRF_K + rank))
            lookup[key] = doc
        for rank, doc in enumerate(bm25_docs):
            key = doc.metadata.get("chunk_id", doc.page_content)
            scores[key] = scores.get(key, 0.0) + self.weights[1] * (1.0 / (RRF_K + rank))
            lookup[key] = doc

        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        return [lookup[k] for k, _ in ranked[:top_n]]

    def clear_cache(self):
        self._bm25_cache.clear()

# Vectorstore 및 HybridRetriever 초기화
vectorstore = Chroma(persist_directory=CHROMA_LOCAL, embedding_function=embeddings)
hybrid_retriever = FilteredHybridRetriever(vectorstore=vectorstore, tokenize_func=kiwi_tokenize, k=30)

# [개선 2] 동일 91개 항목 쿼리에 대한 임베딩 재사용 전역 캐시
_query_embedding_cache: dict[str, list[float]] = {}


In [7]:
# ------------------------------------------------------------------------------
# 셀 5️⃣: 위험 문서 (RISKY_DOCS) 체크 로직
# ------------------------------------------------------------------------------
RISKY_DOCS: dict[tuple, float] = {}
if os.path.exists(EXTRACTION_DIAG_CSV):
    _diag = pd.read_csv(EXTRACTION_DIAG_CSV, dtype={"company_code": str})
    _risky = _diag[_diag["issue_ratio_after"] > 0.3]
    for _, r in _risky.iterrows():
        _code = str(r["company_code"]).zfill(6)
        _year = int(r["year"])
        _doc_type = r["doc_type"]
        RISKY_DOCS[(_code, _year, _doc_type)] = r["issue_ratio_after"]
    print(f"⚠️ 위험 관리 대상 문서 {len(RISKY_DOCS)}건 로드 완료")


⚠️ 위험 관리 대상 문서 0건 로드 완료


In [8]:
# ------------------------------------------------------------------------------
# 셀 6️⃣: 항목별 데이터 구조 생성 및 채점용 Prompt 구성
# ------------------------------------------------------------------------------
def load_rubric_items(filepath, target_area):
    wb = openpyxl.load_workbook(filepath, data_only=True)
    wm = wb[MASTER_SHEET]
    wl = wb[LOOKUP_SHEET]

    mh = [c.value for c in wm[HEADER_ROW]]
    mi = {str(h).strip(): i for i, h in enumerate(mh) if h}
    lh = [c.value for c in wl[HEADER_ROW]]
    li = {str(h).strip(): i for i, h in enumerate(lh) if h}

    lookup = {}
    for row in wl.iter_rows(min_row=DATA_START_ROW, values_only=True):
        c = row[li["분류코드"]]
        if c is None: continue
        lookup.setdefault(c, []).append({
            "단계번호": row[li["단계번호"]],
            "배점": row[li["배점"]],
            "판단기준": row[li["단계별 판단기준(K-ESG v2.0 원문)"]]
        })

    items = []
    for rno, row in enumerate(wm.iter_rows(min_row=DATA_START_ROW, values_only=True), start=DATA_START_ROW):
        code_id = row[mi["분류코드"]]
        if code_id is None: continue
        applic = str(row[mi["적용구분"]])
        area = row[mi["영역"]]
        is_semi = "반도체" in applic

        if target_area == "반도체특화":
            if not is_semi: continue
        else:
            if is_semi: continue
            if area != target_area: continue

        # [개선 4] 검색 쿼리 표준화: 진단항목명 + 실사 진단 질문
        item_name = str(row[mi["진단항목명"]]) if row[mi["진단항목명"]] else ""
        audit_q = str(row[mi["실사 진단 질문"]]) if row[mi["실사 진단 질문"]] else item_name
        search_query = f"{item_name} {audit_q}".strip()

        items.append({
            "row_no": rno,
            "분류코드": code_id,
            "적용구분": applic,
            "영역": area,
            "진단항목명": item_name,
            "채점유형": row[mi["채점유형"]],
            "점검기준상세": row[mi["점검기준 상세(K-ESG v2.0 원문)"]] or "",
            "실사진단질문": audit_q,
            "search_query": search_query,
            "lookup_rows": lookup.get(code_id, [])
        })
    return items

def gemini_judge(prompt: str, keys: list):
    resp = client.models.generate_content(
        model=JUDGE_MODEL, contents=prompt,
        config=types.GenerateContentConfig(temperature=0, response_mime_type="application/json")
    )
    try:
        return json.loads(resp.text)
    except (json.JSONDecodeError, TypeError):
        return {k: None for k in keys}


In [9]:
# ------------------------------------------------------------------------------
# 셀 7️⃣: 단일 항목 채점 실행 함수 (검색, 캐싱, 경고, LLM 평가, 상세 근거 인용)
# ------------------------------------------------------------------------------

def _doc_label(company_name: str, year: int, doc_type: str) -> str:
    """비고란에 적을 근거 문서 이름 (예: SK하이닉스_2023_지속가능경영보고서)."""
    return f"{company_name}_{year}_{DOC_TYPE_LABEL.get(doc_type, doc_type)}"


def _build_numbered_evidence(docs, company_name: str, year: int) -> str:
    """각 근거 청크를 '[문서명] p.페이지' 헤더 + 줄번호가 매겨진 문장으로 구성.
    LLM이 실제로 인용한 줄을 정확히 짚어낼 수 있도록 줄 단위로 번호를 붙여서 준다."""
    blocks = []
    for d in docs:
        label = _doc_label(company_name, year, d.metadata.get("doc_type"))
        page = d.metadata.get("page")
        lines = [ln for ln in d.page_content.split("\n") if ln.strip()]
        numbered = "\n".join(f"{i}: {ln}" for i, ln in enumerate(lines, start=1))
        blocks.append(f"[{label}] p.{page}\n{numbered}")
    return "\n---\n".join(blocks)


def _format_citations(citations) -> str:
    """근거인용 리스트를 비고란 양식으로 변환:
    [문서명]
    p.페이지번호.
    문장(N번째 줄)
    여러 근거는 빈 줄로 구분해서 모두 나열."""
    if not citations:
        return ""
    blocks = []
    for c in citations:
        if not isinstance(c, dict):
            continue
        doc = c.get("문서명") or "출처 불명"
        page = c.get("페이지", "?")
        line = c.get("줄번호", "?")
        blocks.append(f"[{doc}]\np.{page}.\n문장({line}번째 줄)")
    return "\n\n".join(blocks)


CITATION_INSTRUCTION = """
[출처 표기 규칙 — 반드시 지킬 것]
위 근거 자료는 "[문서명] p.페이지번호" 헤더 아래에 줄마다 번호가 매겨져 있습니다.
판단에 실제로 사용한 문장마다 그 문서명, 페이지번호, 줄번호를 근거 자료에 적힌 그대로
"근거인용" 배열에 담으세요. 근거가 여러 문서/여러 줄에 걸쳐 있으면 항목을 나눠서 전부 적고,
지어내지 말고 근거 자료에 실제로 있는 문서명/페이지/줄번호만 쓰세요.
"근거인용": [{"문서명": "<위 헤더의 문서명 그대로>", "페이지": <정수>, "줄번호": <정수>}, ...]
"""


def score_item(item, company_code: str, company_name: str, year: int, is_c261: bool):
    company_code = str(company_code).zfill(6)
    if "반도체" in str(item["적용구분"]) and not is_c261:
        return {"skip": True}

    filter_dict = {"$and": [{"company_code": company_code}, {"year": year}]}

    # [개선 2] 쿼리 임베딩 캐싱 적용
    code_id = item["분류코드"]
    if code_id not in _query_embedding_cache:
        _query_embedding_cache[code_id] = embeddings.embed_query(item["search_query"])
    q_embed = _query_embedding_cache[code_id]

    # 검색 수행
    docs = hybrid_retriever.invoke(
        query=item["search_query"], filter=filter_dict, top_n=5, query_embedding=q_embed
    )

    if not docs:
        base = {"skip": False, "evidence_pages": [], "citation_note": "", "needs_review": True, "review_reason": "근거 미검색"}
        if item["채점유형"] in ("단계형", "선택형"): base["stage"] = 1
        elif item["채점유형"] == "감점형": base["violations"] = {"유형1": 0, "유형2": 0, "유형3": 0}
        else: base["composite_score"] = 1
        return base

    # [개선 3] 추출 부실 위험 문서 경고 감지
    _risky_flag = False
    for d in docs:
        _dtype = d.metadata.get("doc_type")
        if (company_code, year, _dtype) in RISKY_DOCS:
            _risky_flag = True

    ev_text = _build_numbered_evidence(docs, company_name, year)
    ev_pages = sorted({d.metadata.get("page") for d in docs if d.metadata.get("page") is not None})

    # LLM 판단
    if item["채점유형"] in ("단계형", "선택형"):
        detail = "\n".join(f"- {r['단계번호']}단계({r['배점']}점): {r['판단기준']}" for r in item["lookup_rows"])
        prompt = f"""당신은 ESG 진단 전문가입니다. 아래 근거 자료만으로 판단하세요.
근거가 불충분하면 가장 낮은 단계를 선택하세요. 추측 금지.
[진단 항목] {item['진단항목명']}
[K-ESG 판단 기준 원문] {item['점검기준상세']}
[단계별 상세 기준]
{detail}
[근거 자료]
{ev_text}
{CITATION_INSTRUCTION}
몇 단계입니까? JSON: {{"단계번호": <정수>, "근거요약": "<1~2문장>", "근거인용": [...]}}"""
        r = gemini_judge(prompt, ["단계번호", "근거요약", "근거인용"])
        return {
            "skip": False, "stage": r.get("단계번호", 1) or 1, "evidence_pages": ev_pages,
            "citation_note": _format_citations(r.get("근거인용")),
            "needs_review": _risky_flag, "review_reason": "추출 부실 문서 포함 - 재검토 필요" if _risky_flag else None
        }

    elif item["채점유형"] == "감점형":
        prompt = f"""당신은 ESG 위반 이력 진단 전문가입니다. 아래 근거 자료만으로 판단하세요.
언급이 없으면 0건. 추측 금지.
[진단 항목] {item['진단항목명']}
[위반유형 기준 원문] {item['점검기준상세']}
[근거 자료]
{ev_text}
{CITATION_INSTRUCTION}
JSON: {{"유형1_건수":<정수>,"유형2_건수":<정수>,"유형3_건수":<정수>, "근거인용": [...]}}"""
        r = gemini_judge(prompt, ["유형1_건수", "유형2_건수", "유형3_건수", "근거인용"])
        v = {"유형1": r.get("유형1_건수", 0) or 0, "유형2": r.get("유형2_건수", 0) or 0, "유형3": r.get("유형3_건수", 0) or 0}
        return {
            "skip": False, "violations": v, "evidence_pages": ev_pages,
            "citation_note": _format_citations(r.get("근거인용")),
            "needs_review": any(x > 0 for x in v.values()) or _risky_flag,
            "review_reason": "위반 발견 또는 위험 문서 - 재확인 권장" if (any(v.values()) or _risky_flag) else None
        }

    else:
        prompt = f"""당신은 ESG 진단 전문가입니다. 아래 근거 자료만으로 판단하세요.
[진단 항목] {item['진단항목명']}
[세부요소 및 판단기준 원문 - 각 요소 균등가중] {item['점검기준상세']}
[근거 자료]
{ev_text}
{CITATION_INSTRUCTION}
세부요소별 배점(0~100)을 판단하세요. JSON: {{"세부요소":[{{"요소명":"...","배점":<0~100>}}, ...], "근거인용": [...]}}"""
        r = gemini_judge(prompt, ["세부요소", "근거인용"])
        sub = r.get("세부요소") or []
        score = max(1, round(sum(e.get("배점", 0) for e in sub) / len(sub), 1)) if sub else 1
        return {
            "skip": False, "composite_score": score, "evidence_pages": ev_pages,
            "citation_note": _format_citations(r.get("근거인용")),
            "needs_review": True, "review_reason": "복합가중형/위험문서 확인"
        }


In [10]:
# ------------------------------------------------------------------------------
# 셀 8️⃣: 영역별 채점 및 저장 함수 (`score_area`)
# ------------------------------------------------------------------------------
def score_area(target_area: str, company_code: str, year: int, company_name: str, is_c261: bool, rubric_out: str, results_dir: str):
    if not os.path.exists(rubric_out):
        shutil.copy(RUBRIC_SRC, rubric_out)

    items = load_rubric_items(RUBRIC_SRC, target_area)
    wb_out = openpyxl.load_workbook(rubric_out)
    ws_out = wb_out[MASTER_SHEET]
    log = []

    for item in tqdm(items, desc=f"[{company_name}] {target_area}"):
        res = score_item(item, company_code=company_code, company_name=company_name, year=year, is_c261=is_c261)
        r = item["row_no"]

        if res.get("skip"):
            ws_out[f"X{r}"] = "N/A(업종 미해당)"
            log.append({"분류코드": item["분류코드"], "결과": "스킵"})
            continue

        if item["채점유형"] in ("단계형", "선택형"):
            ws_out[f"T{r}"] = res["stage"]
        elif item["채점유형"] == "감점형":
            v = res["violations"]
            ws_out[f"U{r}"] = v["유형1"]
            ws_out[f"V{r}"] = v["유형2"]
            ws_out[f"W{r}"] = v["유형3"]
        else:
            ws_out[f"X{r}"] = res["composite_score"]

        # [상세 근거 인용] 비고란 = 근거 문서명/페이지/줄번호 인용 블록 + (필요 시) 검토 태그.
        # 이 두 가지를 빈 줄로 구분해서 전부 적는다 — 근거를 대충 요약하지 않는 게 핵심.
        note = []
        if res.get("citation_note"): note.append(res["citation_note"])
        if res.get("needs_review"): note.append(f"[AI검토필요]{res.get('review_reason','')}")
        if note: ws_out[f"Z{r}"] = "\n\n".join(note)

        log.append({
            "분류코드": item["분류코드"],
            "채점유형": item["채점유형"],
            "결과": res.get("stage") or res.get("composite_score") or res.get("violations"),
            "근거페이지": res.get("evidence_pages"),
            "근거인용": res.get("citation_note"),
            "검토필요": res.get("needs_review")
        })

    wb_out.save(rubric_out)
    df_log = pd.DataFrame(log)
    log_csv = f"{results_dir}/{company_code}_{year}_{target_area}_채점로그.csv"
    df_log.to_csv(log_csv, index=False, encoding="utf-8-sig")
    return df_log


In [11]:
# ------------------------------------------------------------------------------
# 셀 9️⃣: [SK하이닉스 전용] 반도체특화 포함 전체 연도 순차 자동 채점
# ------------------------------------------------------------------------------
# SK하이닉스 전용 Chroma(CHROMA_LOCAL)에 실제로 인덱싱된 연도를 자동으로 뽑아서,
# 그 연도들에 대해 환경/사회/지배구조/정보공시 + 반도체특화(IS_C261=True)까지 채점한다.
_GET_PAGE_SIZE = 5000
_all_metadatas = []
_offset = 0
while True:
    _page = vectorstore.get(include=["metadatas"], limit=_GET_PAGE_SIZE, offset=_offset)
    _metas = _page["metadatas"]
    if not _metas: break
    _all_metadatas.extend(_metas)
    _offset += len(_metas)

_meta_df = pd.DataFrame(_all_metadatas)
_meta_df = _meta_df[_meta_df["company_code"].astype(str).str.zfill(6) == COMPANY_CODE]
if _meta_df.empty:
    raise ValueError(
        f"{CHROMA_LOCAL} 안에 company_code={COMPANY_CODE} 청크가 없음 — "
        "build_rag_index_v6_production_SKHY.ipynb로 먼저 재인덱싱을 끝낼 것"
    )

target_years = sorted(_meta_df["year"].unique().tolist())
print(f"🚀 {COMPANY_NAME}({COMPANY_CODE}) — 반도체특화 포함, 총 {len(target_years)}개 연도 순차 채점: {target_years}\n")


def _area_done(results_dir: str, company_code: str, year: int, area: str) -> bool:
    """해당 영역의 채점 로그 CSV가 이미 있으면(= score_area가 끝까지 성공해서 저장됨) 완료로 간주."""
    return os.path.exists(f"{results_dir}/{company_code}_{year}_{area}_채점로그.csv")


results_dir = f"{REPORTS_ROOT}/{COMPANY_NAME}({COMPANY_CODE})/results"
os.makedirs(results_dir, exist_ok=True)

for year in target_years:
    rubric_out = f"{results_dir}/{COMPANY_CODE}_{year}_채점완료_전체.xlsx"

    # [SK하이닉스 전용] IS_C261=True라서 반도체특화까지 area_order에 포함됨
    area_order = ["환경", "사회", "지배구조", "정보공시"] + (["반도체특화"] if IS_C261 else [])
    _remaining_areas = [a for a in area_order if not _area_done(results_dir, COMPANY_CODE, year, a)]

    if not _remaining_areas:
        print(f"⏭️  {COMPANY_NAME}({COMPANY_CODE}) {year}년 — 이미 전체 완료, 스킵")
        continue

    print(f"\n============================================================")
    print(f"▶ {COMPANY_NAME}({COMPANY_CODE}) / {year}년 — 남은 영역: {_remaining_areas}")
    print(f"============================================================")

    hybrid_retriever.clear_cache()

    for area in _remaining_areas:
        score_area(
            target_area=area,
            company_code=COMPANY_CODE,
            year=year,
            company_name=COMPANY_NAME,
            is_c261=IS_C261,
            rubric_out=rubric_out,
            results_dir=results_dir
        )

    print(f"✅ {COMPANY_NAME}({COMPANY_CODE}) {year}년 채점 완료 → 파일 저장: {rubric_out}")

print(f"\n🎉 {COMPANY_NAME}({COMPANY_CODE}) 전체 연도 순차 ESG 채점이 모두 끝났습니다!")


🚀 SK하이닉스(000660) — 반도체특화 포함, 총 4개 연도 순차 채점: [2022, 2023, 2024, 2025]


▶ SK하이닉스(000660) / 2022년 — 남은 영역: ['환경', '사회', '지배구조', '정보공시', '반도체특화']


[SK하이닉스] 반도체특화: 100%|██████████| 6/6 [00:09<00:00,  1.55s/it]


✅ SK하이닉스(000660) 2022년 채점 완료 → 파일 저장: /Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/05_채점 RAG/260902_RAG테스트/보고서 3종/SK하이닉스(000660)/results/000660_2022_채점완료_전체.xlsx

▶ SK하이닉스(000660) / 2023년 — 남은 영역: ['환경', '사회', '지배구조', '정보공시', '반도체특화']


[SK하이닉스] 반도체특화: 100%|██████████| 6/6 [00:08<00:00,  1.34s/it]


✅ SK하이닉스(000660) 2023년 채점 완료 → 파일 저장: /Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/05_채점 RAG/260902_RAG테스트/보고서 3종/SK하이닉스(000660)/results/000660_2023_채점완료_전체.xlsx

▶ SK하이닉스(000660) / 2024년 — 남은 영역: ['환경', '사회', '지배구조', '정보공시', '반도체특화']


[SK하이닉스] 반도체특화: 100%|██████████| 6/6 [00:08<00:00,  1.48s/it]


✅ SK하이닉스(000660) 2024년 채점 완료 → 파일 저장: /Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/05_채점 RAG/260902_RAG테스트/보고서 3종/SK하이닉스(000660)/results/000660_2024_채점완료_전체.xlsx

▶ SK하이닉스(000660) / 2025년 — 남은 영역: ['환경', '사회', '지배구조', '정보공시', '반도체특화']


[SK하이닉스] 반도체특화: 100%|██████████| 6/6 [00:08<00:00,  1.38s/it]


✅ SK하이닉스(000660) 2025년 채점 완료 → 파일 저장: /Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/05_채점 RAG/260902_RAG테스트/보고서 3종/SK하이닉스(000660)/results/000660_2025_채점완료_전체.xlsx

🎉 SK하이닉스(000660) 전체 연도 순차 ESG 채점이 모두 끝났습니다!
